# Ricerca Informata ed Euristiche

In questa lezione di tutoraggio esploreremo la differenza tra grafi dello spazio degli stati e alberi di ricerca, per poi metterci al lavoro e provare a sviluppare alcune euristiche per risolvere un problema "semplice".

## Grafi dello Spazio degli Stati vs. Alberi di Ricerca

### Grafi dello Spazio degli Stati
Un **Grafo dello Spazio degli Stati** rappresenta il mondo fisico o lo spazio di configurazione reale del problema.
* Ogni stato unico (es. lo stato C) esiste **una sola volta**.
* Percorsi distinti possono portare allo stesso stato fisico (es. raggiungere lo stato C tramite il percorso $S \to A \to C$ oppure $S \to B \to C$).

### Alberi di Ricerca
Un **Albero di Ricerca** srotola le scelte del grafo nel tempo, mappando tutte le possibili sequenze di azioni a partire dallo stato iniziale.
* Se usiamo una **Tree Search** pura, stati identici vengono trattati come nodi completamente distinti se raggiunti tramite percorsi diversi.
* Questo causa espansioni ridondanti e una crescita esponenziale del numero di nodi se il grafo contiene cicli o percorsi convergenti.
* La **Graph Search** risolve questa inefficienza mantenendo un insieme `explored` (closed list) per potare le valutazioni duplicate degli stati.

![SSG vs ST](./imgs/ssg_vs_st.png)

## Euristiche

Una **euristica** $h(n)$ è una funzione specifica del dominio che stima il costo del percorso più economico da uno stato dato $n$ all'obiettivo. Funziona come una bussola informata per algoritmi di ricerca come A*.

### Il problema con UCS (Uniform Cost Search)
UCS espande i nodi basandosi unicamente sul costo del percorso $g(n)$. Pur garantendo di trovare il percorso ottimo, esplora ciecamente in tutte le direzioni come una macchia d'olio che si espande, portando a un numero enorme di espansioni man mano che la profondità di ricerca cresce.

### Tipi di euristiche e problemi rilassati
Tipicamente costruiamo euristiche ammissibili **rilassando** i vincoli del problema (rendendo le regole più semplici):
1. **Euristiche ammissibili:** un'euristica è ammissibile se non sovrastima mai il costo reale per raggiungere l'obiettivo ($h(n) \le h^*(n)$). Questo garantisce l'ottimalità nella ricerca A* su albero.
2. **Euristiche consistenti (monotone):** soddisfano la disuguaglianza triangolare ($h(n) \le cost(n, n') + h(n')$), garantendo l'ottimalità nella Graph Search senza dover riaprire nodi già esplorati.

## Mettiamoci al lavoro: il puzzle

### Logica e difficoltà dell'8-Puzzle
L'**8-Puzzle** è costituito da una griglia 3x3 con 8 tessere numerate e uno spazio vuoto (rappresentato da `0`). Le tessere possono scorrere in orizzontale o in verticale nello spazio vuoto.

**Difficoltà nel risolverlo:**
* La dimensione dello spazio degli stati è $9! / 2 = 181.440$ configurazioni raggiungibili.
* Le ricerche cieche si impantanano facilmente esplorando sotto-alberi profondi senza un senso della direzione, con il risultato di migliaia o milioni di nodi espansi.

![Puzzle1](./imgs/puzzle1.png)
![Puzzle2](./imgs/puzzle2.png)

Qui abbiamo una semplice implementazione dello stato del puzzle. Useremo questo oggetto per risolvere il problema.

Giusto per darti chiarezza e aiutarti a sviluppare la classe, ecco una spiegazione di cosa rappresenta lo stato e cosa dovrebbero fare i metodi.

* `__init__`: inizializza la tupla della scacchiera, tenendo traccia del nodo padre, dell'azione eseguita per raggiungerla, del costo del percorso $g$, del valore euristico $h$ e del punteggio totale di valutazione $f = g + h$.
* `state_key`: restituisce una rappresentazione unica e hashable (la tupla della scacchiera) per l'insieme `explored` della graph search.
* `get_neighbors`: individua la tessera vuota (`0`), calcola i movimenti validi di riga/colonna (`UP`, `DOWN`, `LEFT`, `RIGHT`) e genera tutti gli stati successori legali scambiando le tessere.

In [ ]:
import heapq

class PuzzleState:
    def __init__(self, board, parent=None, action=None, cost=0, h=0):
        self.board = board       # Tupla che rappresenta la scacchiera 3x3, es. (7, 2, 4, 5, 0, 6, 8, 3, 1)
        self.parent = parent     # Nodo padre (PuzzleState)
        self.action = action     # Azione eseguita per raggiungere questo stato
        self.g = cost            # Costo del percorso (numero di passi effettuati)
        self.h = h               # Valore euristico
        self.f = self.g + self.h # Punteggio totale f = g + h

    @property
    def state_key(self):
        # Restituisco la tupla della scacchiera come chiave unica e hashable
        return self.board

    def get_neighbors(self):
        neighbors = []
        # 1. Trovo l'indice della tessera vuota (0)
        idx = self.board.index(0)
        row, col = divmod(idx, 3)
        
        # 2. Definisco i possibili movimenti: Su, Giù, Sinistra, Destra
        moves = {
            "UP": (-1, 0),
            "DOWN": (1, 0),
            "LEFT": (0, -1),
            "RIGHT": (0, 1)
        }
        
        for action, (dr, dc) in moves.items():
            # 3. Calcolo la nuova posizione e verifico che sia dentro la griglia
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_idx = new_row * 3 + new_col
                # 4. Scambio la tessera vuota con la tessera vicina
                board_list = list(self.board)
                board_list[idx], board_list[new_idx] = board_list[new_idx], board_list[idx]
                neighbors.append((tuple(board_list), action))
                
        return neighbors

È ora di scrivere le nostre euristiche! Ricorda:

1. **`h_zero` (UCS / $h = 0$):** nessuna guida euristica. Riduce la ricerca A* alla Uniform Cost Search, causando un'esplorazione pesante.
2. **`h_misplaced_tiles` (Tiles / $h_1$):** conta quante tessere sono fuori posto rispetto all'obiettivo (ignorando la tessera vuota). Basata sul problema rilassato in cui qualsiasi tessera può saltare in qualsiasi posizione.
3. **`h_manhattan` (Distanza di Manhattan / $h_2$):** somma le distanze assolute orizzontali e verticali sulla griglia di ogni tessera dalla sua posizione target. Basata sul problema rilassato in cui le tessere possono muoversi indipendentemente l'una attraverso l'altra.

In [ ]:
# --- Funzioni euristiche ---
def h_zero(state, goal):
    """UCS: nessuna guida euristica (h = 0)"""
    return 0

def h_misplaced_tiles(state, goal):
    """Euristica Tiles: numero di tessere fuori posto (esclusa la cella vuota 0)"""""
   
   # TODO
   # Conto qunte tessere sono fuori posto, ignorando lo 0
    count = 0
    for s, g in zip(state, goal):

        # Se la tessera corrente non è lo 0, la conto
        if s != 0 and s != g:
            count += 1
    return count


def h_manhattan(state, goal):
    """Euristica Distanza di Manhattan: somma delle distanze sulla griglia per ogni tessera"""

    # TODO
    # Costruisco una mappa valore -> (riga, colonna) per la scacchiera obiettivo
    goal_pos = {val: divmod(i, 3) for i, val in enumerate(goal)}

    # Accumulo la distanza di Manhattan di ogni tessera dalla sua posizione target
    total = 0
    for i, val in enumerate(state):
        if val == 0:
            # Ignoro la cella vuota
            continue
        r, c = divmod(i, 3)
        gr, gc = goal_pos[val]
        total += abs(r - gr) + abs(c - gc)
    return total

Mettiamo poi insieme tutto per scrivere il solver finale del puzzle.\
La funzione `solve_puzzle` implementa una coda a priorità (`heapq`) ordinata per punteggio di valutazione $f(n) = g(n) + h(n)$:
* Estrae il nodo con costo minore, verifica se coincide con il `goal_board` e, in caso affermativo, ricostruisce il percorso delle azioni.
* In base al parametro `search_type`, alterna tra **Graph Search** (usando un insieme `explored` per filtrare le valutazioni duplicate) e **Tree Search**.

In [ ]:
def solve_puzzle(start_board, goal_board, heuristic_func, search_type="graph"):
    # 1. Creo il nodo iniziale con g=0 e h calcolata dall'euristica
    start_node = PuzzleState(
        board=start_board, 
        cost=0, 
        h=heuristic_func(start_board, goal_board)
    )
    
    # 2. Inizializzo la coda a priorità ordinata per f(n) = g(n) + h(n)
    frontier = []
    counter = 0  # tie-breaker per l'heap
    heapq.heappush(frontier, (start_node.f, counter, start_node))
    
    # 3. Inizializzo l'insieme degli stati esplorati e i contatori
    explored = set()
    nodes_expanded = 0
    max_frontier_size = 1

    while frontier:
        # 4. Estraggo il nodo con f minore
        _, _, current = heapq.heappop(frontier)

        # 5. Test dell'obiettivo: se la scacchiera coincide con il goal, ricostruisco il percorso
        if current.board == goal_board:
            path = []
            # Risalgo i puntatori parent fino alla radice
            node = current
            while node.parent is not None:
                path.append((node.action, node.board))
                node = node.parent
            path.reverse()
            return {
                "success": True,
                "path_length": len(path),
                "nodes_expanded": nodes_expanded,
                "max_frontier": max_frontier_size,
                "path": path,
            }
            
        # 6. Nella Graph Search salto gli stati già esplorati;
        if search_type == "graph":
            if current.state_key in explored:
                continue
            explored.add(current.state_key)
            
        # 7. Incremento il contatore dei nodi espansi
        nodes_expanded += 1
        
        # 8. Espando i vicini e li aggiungo alla frontiera
        for neighbor_board, action in current.get_neighbors():
            # Costo del figlio: g del padre + 1 (ogni passo costa 1)
            child_g = current.g + 1
            # Euristica del figlio
            child_h = heuristic_func(neighbor_board, goal_board)
            # Creo il nodo figlio
            child = PuzzleState(
                board=neighbor_board,
                parent=current,
                action=action,
                cost=child_g,
                h=child_h,
            )
            # Inserisco nella coda a priorità con il contatore come tie-breaker
            counter += 1
            heapq.heappush(frontier, (child.f, counter, child))
            # Aggiorno la dimensione massima della frontiera
            if len(frontier) > max_frontier_size:
                max_frontier_size = len(frontier)
            
    return {"success": False, "nodes_expanded": nodes_expanded}

Qui abbiamo una semplice funzione di animazione per visualizzare la soluzione trovata.

Puoi provare a usare dfs/bfs o qualche euristica strana (non corretta) e vedere quale soluzione trovano.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
from IPython.display import HTML

def create_puzzle_gif(start_board, path, filename="puzzle_solution.gif"):
    """
    Crea e restituisce un GIF animato del percorso di soluzione dell'8-puzzle.
    """
    # 1. Costruisco il percorso completo includendo lo stato iniziale
    full_path = [(None, start_board)] + path
    
    fig, ax = plt.subplots(figsize=(4, 4))
    
    def update(frame_idx):
        # 2. Pulisco l'asse e rimuovo i tick
        ax.clear()
        ax.set_xticks([])
        ax.set_yticks([])
        
        # 3. Estraggo azione e scacchiera del frame corrente
        action, board = full_path[frame_idx]
        grid = np.array(board).reshape(3, 3)
        
        # 4. Disegno le 9 celle con i colori appropriati
        for r in range(3):
            for c in range(3):
                val = grid[r, c]
                # Colore di sfondo diverso per la cella vuota
                bg_color = "#333333" if val == 0 else "#e1f5fe"
                text_color = "white" if val == 0 else "#01579b"
                text_val = str(val) if val != 0 else ""
                
                ax.add_patch(plt.Rectangle((c, 2 - r), 1, 1, facecolor=bg_color, edgecolor="black", linewidth=1.5))
                # 5. Scrivo il numero al centro della cella (tranne per lo 0)
                if val != 0:
                    ax.text(c + 0.5, 2 - r + 0.5, text_val, fontsize=20, fontweight='bold', 
                            ha='center', va='center', color=text_color)
                    
        ax.set_xlim(0, 3)
        ax.set_ylim(0, 3)
        ax.set_aspect('equal')
        
        # 6. Imposto il titolo: "Start State" per il primo frame, altrimenti azione e step
        if frame_idx == 0:
            ax.set_title("Stato Iniziale", fontsize=14, fontweight='bold', pad=10)
        else:
            ax.set_title(f"Step {frame_idx} | Azione: {action}", fontsize=12, pad=10)

    # 7. Creo l'animazione e la salvo come GIF
    anim = animation.FuncAnimation(fig, update, frames=len(full_path), interval=1000, repeat=True)
    
    anim.save(filename, writer='pillow', fps=1)
    plt.close(fig)

    # 8. Restituisco l'HTML per visualizzare il GIF nel notebook
    return HTML(f'<img src="{filename}" alt="Puzzle Animation">')

E qui abbiamo il nostro confronto finale.\
Sentiti libero di cambiare lo stato iniziale, lo stato obiettivo, le euristiche ecc. L'immaginazione è il tuo limite.\
Ti suggerisco anche di fare un po' di profiling aggiungendo dei contatori di tempo, come abbiamo già fatto nelle lezioni precedenti.

In [ ]:
# 1. Definisco lo stato iniziale e lo stato obiettivo
start_state = (7, 2, 4, 
               5, 0, 6, 
               8, 3, 1)

goal_state  = (0, 1, 2, 
               3, 4, 5, 
               6, 7, 8)

# 2. Risolvo con UCS (h = 0), Tiles e Manhattan in Graph Search
res_ucs = solve_puzzle(start_state, goal_state, h_zero, search_type="graph")

res_tiles = solve_puzzle(start_state, goal_state, h_misplaced_tiles, search_type="graph")

res_manhattan = solve_puzzle(start_state, goal_state, h_manhattan, search_type="graph")

# 3. Mostro l'animazione della soluzione trovata da Manhattan
if res_manhattan["success"]:
    display(create_puzzle_gif(start_state, res_manhattan["path"]))
else:
    print("Nessuna soluzione trovata.")

print()

# 4. Stampo i risultati per ciascuna strategia
print(f"Strategia: UCS (Ricerca Cieca)\n    Nodi espansi : {res_ucs['nodes_expanded']}\n    Lunghezza del percorso : {res_ucs['path_length']}")
print(f"Strategia: Misplaced Tiles (h1)\n    Nodi espansi : {res_tiles['nodes_expanded']}\n    Lunghezza del percorso : {res_tiles['path_length']}")
print(f"Strategia: Manhattan Distance (h2)\n    Nodi espansi : {res_manhattan['nodes_expanded']}\n    Lunghezza del percorso : {res_manhattan['path_length']}")